# HP-tuning stability check

Does it matter where the 10-day tuning block sits?  
For a subset of stocks the tune → freeze → walk-forward pipeline is run with the block at 0%, 25%, 50% and 75% of the sample, evaluating on the days after the block.

In [1]:
import os, sys, json, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_processing as du
import utils.execution as execution
import utils.pipeline as pipeline
import utils.workers as workers

for m in (du, pipeline, execution, workers):
    importlib.reload(m)

In [2]:
# Config (same tuning protocol as xgboost.ipynb, plus OFFSETS)

STABILITY_SYMBOLS = ['Siemens','Adidas', 'RWE', 'Continental', 'Beiersdorf']

OFFSETS = [0.00, 0.33, 0.67] # tuning-block start

TUNE_BLOCK_LEN = 10
TRAIN_DAYS = 1

N_TRIALS = 40
N_PAIRS = 5

SEED = 0

N_PROC = 5
N_JOBS = 2

FEATURES = ["L1-QDiff", "MicroPrice"]
HORIZONS = ["100ms","2s", "30s", "2.5m"]

RUN_NAME = "1-days_standard"

DM_REGRESSION_RUN_ID = "1-days_standard"

# name -> (distribution, low, high)
SEARCH_SPACE = {
    "max_depth": ("int", 1, 6),
    "learning_rate": ("log", 0.01, 0.3),
    "min_child_weight": ("logint", 5, 100),
    "subsample": ("uniform", 0.5, 1.0),
    "colsample_bytree": ("uniform", 0.5, 1.0),
    "reg_lambda": ("log", 0.01, 10.0),
}

In [3]:
# XGB Params
EARLY_STOPPING = dict(n_estimators=2000, early_stopping_rounds=50, eval_metric="rmse")

XGB_PARAMS = dict(
    tree_method="hist",
    max_bin=128,
    n_jobs=N_JOBS,
    random_state=0,
)

DEVICE = execution.select_device()
print(f"XGBoost device: {DEVICE}")

# Tuning block and eval window per offset
PARENT = os.path.dirname(os.getcwd())
MODEL_ROOT = f"{PARENT}/model_outputs/XGBoost"
STABILITY_DIR = f"{MODEL_ROOT}/runs/{RUN_NAME}"

FEATURE_COLS, TARGET_COLS = workers.feature_target_cols(STABILITY_SYMBOLS[0], HORIZONS, FEATURES)

ALL_DATES = list(du.SAMPLE_DATES)
BLOCKS = {}
print(f"totals: trials {len(TARGET_COLS)} targets/stock (every offset)")
for off in OFFSETS:
    start = round(off * len(ALL_DATES))
    tune_dates = ALL_DATES[start:start + TUNE_BLOCK_LEN]
    eval_dates = ALL_DATES[start + TUNE_BLOCK_LEN:]
    if len(tune_dates) < TUNE_BLOCK_LEN or len(eval_dates) < 2:
        raise ValueError(f"offset {off:.0%}: not enough days (tune {len(tune_dates)}, eval {len(eval_dates)})")
    BLOCKS[off] = {"tune_dates": tune_dates, "eval_dates": eval_dates}
    print(f"{off:>4.0%}: tune {tune_dates[0]} .. {tune_dates[-1]}, "
          f"eval {eval_dates[0]} .. {eval_dates[-1]} ({len(eval_dates)} days), "
          f"partial {len(eval_dates) - TRAIN_DAYS} days/stock")

XGBoost device: cuda:0
totals: trials 20 targets/stock (every offset)
  0%: tune 2023-01-02 .. 2023-01-13, eval 2023-01-16 .. 2023-06-30 (117 days), partial 116 days/stock
 33%: tune 2023-03-01 .. 2023-03-14, eval 2023-03-15 .. 2023-06-30 (75 days), partial 74 days/stock
 67%: tune 2023-05-04 .. 2023-05-17, eval 2023-05-18 .. 2023-06-30 (32 days), partial 31 days/stock


In [ ]:
# Each offset is one run dir with its own manifest.
GLOBAL_START = time.perf_counter()

def update_manifest(run_dir, **fields):
    with open(f"{run_dir}/manifest.json") as f:
        manifest = json.load(f)
    manifest.update(fields)
    with open(f"{run_dir}/manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    return manifest


for off in OFFSETS:
    off_pct = int(off * 100)
    name = f"stability_off_{off_pct}"
    run_dir = f"{STABILITY_DIR}/{name}"
    tune_dates = BLOCKS[off]["tune_dates"]

    if not os.path.exists(f"{run_dir}/manifest.json"):
        # nested name so start_run puts the run dir under STABILITY_DIR
        pipeline.start_run(MODEL_ROOT, f"{RUN_NAME}/{name}", {
            "status": "tuning",
            "purpose": f"HP-tuning stability check, tuning block at {off:.0%} of the sample",
            "offset": off,
            "symbols": STABILITY_SYMBOLS,
            "horizons": HORIZONS,
            "features": FEATURES,
            "feature_cols": FEATURE_COLS,
            "target_cols": TARGET_COLS,
            "train_days": TRAIN_DAYS,
            "tune_dates": list(tune_dates),
            "xgb_params": XGB_PARAMS,
            "target_scale": pipeline.TARGET_SCALE,
            "tuning": {"seed": SEED, "n_trials": N_TRIALS, "n_pairs": N_PAIRS,
                       "search_space": SEARCH_SPACE, "early_stopping": EARLY_STOPPING},
            "params": None,
        })

    # tuning (skips stocks with a trials checkpoint)
    todo = [s for s in STABILITY_SYMBOLS if not os.path.exists(f"{run_dir}/trials/{s}.parquet")]
    execution.run_parallel(workers.tune_xgb, {s: (run_dir, s, DEVICE) for s in todo}, n_proc=N_PROC)

    # freeze winners into the manifest
    all_trials = pd.concat([pd.read_parquet(f"{run_dir}/trials/{s}.parquet") for s in STABILITY_SYMBOLS], ignore_index=True)
    best_params = workers.freeze_winners(all_trials, SEARCH_SPACE)
    update_manifest(run_dir, params=best_params, status="running")

    # walk-forward on the days after the block (skips stocks with a partial checkpoint)
    todo = [s for s in STABILITY_SYMBOLS if not os.path.exists(f"{run_dir}/partial/{s}.parquet")]
    execution.run_parallel(workers.run_xgb, {s: (run_dir, s, DEVICE) for s in todo}, n_proc=N_PROC)

    daily = pd.concat([pd.read_parquet(f"{run_dir}/partial/{s}.parquet") for s in STABILITY_SYMBOLS], ignore_index=True)
    daily.to_parquet(f"{run_dir}/daily_diagnostics.parquet", index=False)
    manifest = update_manifest(run_dir, status="complete")
    update_manifest(run_dir, runtime_seconds=round(
        (datetime.now(timezone.utc) - datetime.fromisoformat(manifest["created_at"])).total_seconds(), 1))
    print(f"{name}: done, {time.perf_counter()-GLOBAL_START:.0f}s elapsed")

# combined file for the analysis cells
daily_results = pd.concat(
    [pd.read_parquet(f"{STABILITY_DIR}/stability_off_{int(off * 100)}/daily_diagnostics.parquet")
       .assign(model_leg="xgb", offset_pct=int(off * 100))
     for off in OFFSETS],
    ignore_index=True,
)
daily_results.to_parquet(f"{STABILITY_DIR}/daily_diagnostics.parquet", index=False)
print(f"TOTAL STABILITY-CHECK TIME: {time.perf_counter()-GLOBAL_START:.2f}s")